# Finite Differences

## Midpoint Method

The midpoint method formula is:

$$y(x+h) = y(x) + hf[x+\frac{h}{2}, y(x) + \frac{h}{2}f(x,y(x))]

To check whether this is accurate to the second order, we need to take the Taylor expansion of a given function, say:

$$ y(x+h) = y(x) +h f (x,y(x)) + \frac{h^2}{2}[\frac{\partial f}{\partial x} + f \frac{\partial f}{\partial y}] + O(h^3)$$

And confirm that with the midpoint method, we can achieve this second order approximation.
$$y(x+h) = y(x) +h f (x,y(x)) + \frac{h^2}{2}[\frac{\partial f}{\partial x} + f \frac{\partial f}{\partial y}]$$

Therefore, if we take the approximation from the midpoint method, then the only remaining term would be $O(h^3)$


We can define the step by step of the midpoint method by:
$$k_1 = f(t_n, y_n), \qquad k_2 = f\!\left(t_n + \tfrac{h}{2},\; y_n + \tfrac{h}{2}k_1\right), \qquad y_{n+1} = y_n + h\,k_2.$$


$$k_2 = f\!\left(t_n + \tfrac{h}{2},\; y_n + \tfrac{h}{2}f\right) = f + \frac{h}{2}f_t + \frac{h}{2}f\,f_y + O(h^2),$$

Therefore:
$$y_{n+1} = y_n + h\,k_2 = y_n + hf + \frac{h^2}{2}f_t + \frac{h^2}{2}f\,f_y + O(h^3). \tag{2}$$


When we use the local truncation error, we see these cancel out exactly

##  Euler and fourth-order Runge-Kutta differential equation methods

To solve this dynamics system problem, we can convert this into a system of differential equations to solve:
$$\ddot{x} + x = 0$$

We can rewrite this to be a first order differential of some form so that:
$$\dot{u} = f(x) = \bmatrix{u_2 // -u_1}$$

Solving this first with Euler:

$$u_{n+1} = u_n + h f(t_n, u_n)$$

Solving with Runge-Kutta:
$$k_1 = hf(x,y(x)) = f(t_n, u_n)$$ 
$$k_2 = hf(x+\frac{h}{2}, y(x) + \frac{k_1}{2}) = f(t_n +\frac{h}{2}, u_n + \frac{h}{2}k_1)$$
$$k_3 = hf(x+\frac{h}{2}, y(x)+ \frac{k_2}{2}) = f(t_n + \frac{h}{2}, u_n + \frac{h}{2}k_2)$$
$$k_4 = hf(x+h, y(x)+k_3) = f(t_n+h,u_n+hk_3)$$

Therefore:
$$u_{n+1} = \frac{h}{6}(k_1+2k_2+2k_3+k_4)$$


In [29]:
import numpy as np

def euler_method(f, t0, T, u0, h, *params):
    t, u = t0, np.asarray(u0, dtype=float)
    ts, us = [t], [u.copy()]
    while t < T - 1e-12:
        u = u + h * np.asarray(f(t, u, *params), dtype=float)
        t += h
        ts.append(t); us.append(u.copy())
    return np.array(ts), np.array(us)


def runge_kutta_method(f, t0, T, u0, h, *params):
    t, u = t0, np.asarray(u0, dtype=float)
    ts, us = [t], [u.copy()]
    while t < T - 1e-12:
        k1 = np.asarray(f(t,       u,          *params), dtype=float)
        k2 = np.asarray(f(t + h/2, u + h/2*k1, *params), dtype=float)
        k3 = np.asarray(f(t + h/2, u + h/2*k2, *params), dtype=float)
        k4 = np.asarray(f(t + h,   u + h*k3,   *params), dtype=float)
        u  = u + h/6 * (k1 + 2*k2 + 2*k3 + k4)
        t += h
        ts.append(t); us.append(u.copy())
    return np.array(ts), np.array(us)

In [32]:
def f_harmonic_oscillator(t, u):
    return np.array([u[1], -u[0]])

t_s, us = runge_kutta_method(f_harmonic_oscillator, 0, 2*np.pi, [1.0, 0.0], h=0.01)
t_s, us = euler_method(f_harmonic_oscillator, 0, 2*np.pi, [1.0, 0.0], h=0.01)
x, x_dot = us[:, 0], us[:, 1]


## Pendulum motion

The pendelums motion is described by:
$$l\ddot{\theta} + (g+\ddot{z})sin(\theta) = 0$$

To numerically solve this system, we can again conver this into a system of two first order differential equations for $\theta$ and $z$ where $u = (\theta, \dot{\theta})$ for $\theta$, and for $z$, we can directly rewrite this from $z(t) = Acos(\omega t)$, which becomes:
$$ \ddot{z} = -A\omega^2 cost(\omega t)$$

Therefore the total system becomes:

$$\dot{\mathbf{u}} = \begin{bmatrix} u_2 \\ -\dfrac{g - A\omega^2\cos(\omega t)}{l}\,\sin u_1 \end{bmatrix}$$

We can reuse the components we made for the first problem here 


$$k_1 = \begin{pmatrix} \dot\theta_n \\ -\frac{g - A\Omega^2\cos(\Omega t_n)}{l}\sin\theta_n \end{pmatrix}$$

$$k_2 = \begin{pmatrix} \dot\theta_n + \frac{h}{2}k_1^{(2)} \\ -\frac{g - A\Omega^2\cos(\Omega(t_n+h/2))}{l}\sin\!\left(\theta_n + \frac{h}{2}k_1^{(1)}\right) \end{pmatrix}$$

and similarly for $k_3$, $k_4$, then:

$$\theta_{n+1} = \theta_n + \frac{h}{6}(k_1^{(1)} + 2k_2^{(1)} + 2k_3^{(1)} + k_4^{(1)}),$$
$$\dot\theta_{n+1} = \dot\theta_n + \frac{h}{6}(k_1^{(2)} + 2k_2^{(2)} + 2k_3^{(2)} + k_4^{(2)}).$$

In [ ]:
g = 9.81
l  = 1

def f_pendulum(t, u, A, Omega):
    z_ddot = -A * Omega**2 * np.cos(Omega * t)
    return np.array([u[1], -(g + z_ddot) / l * np.sin(u[0])])

omega0 = np.sqrt(g / l)

ts, us = runge_kutta_method(f_pendulum, 0, 80, [0.05, 0.0], 0.002, 0.1, 2*omega0)

for Omega_ratio in [1.8, 2.0, 2.2]:
    ts, us = runge_kutta_method(f_pendulum, 0, 80, [0.05, 0.0], 0.002,
                 0.1, Omega_ratio * omega0)
    print(f"Omega/omega0 = {Omega_ratio:.1f}, max|theta| = {np.max(np.abs(us[:,0])):.4f}")



Omega/omega0 = 1.8, max|theta| = 0.0500
Omega/omega0 = 2.0, max|theta| = 1.7351
Omega/omega0 = 2.2, max|theta| = 0.7053


In [36]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
style  = {'description_width': '160px'}
layout = widgets.Layout(width='480px')

w_A     = widgets.FloatSlider(value=0.1,  min=0.0,  max=2.0, step=0.05,
                               description='A', style=style, layout=layout)
w_Omega = widgets.FloatSlider(value=2.0,  min=0.5,  max=4.0, step=0.05,
                               description='Omega/omega0',   style=style, layout=layout)
w_theta0= widgets.FloatSlider(value=0.2,  min=-np.pi, max=np.pi, step=0.05,
                               description='Theta(initial angle, rad)',  style=style, layout=layout)
w_dth0  = widgets.FloatSlider(value=0.0,  min=-5.0, max=5.0,  step=0.1,
                               description='Theta dot(initial velocity)',    style=style, layout=layout)
w_T     = widgets.FloatSlider(value=40.0, min=5.0,  max=200.0, step=5.0,
                               description='T  (integration time, s)',  style=style, layout=layout)
w_method= widgets.ToggleButtons(options=['Runge Kutta', 'Euler'],
                                 description='Method:', style={'description_width': '60px'})

out = widgets.Output()

def run(_=None):
    A     = w_A.value
    Omega = w_Omega.value * omega0
    u0    = [w_theta0.value, w_dth0.value]
    T_end = w_T.value
    h     = 0.005
    solve = runge_kutta_method if w_method.value == 'Runge Kutta' else euler_method

    ts, us = solve(f_pendulum, 0, T_end, u0, h, A, Omega)
    theta  = us[:, 0]
    dtheta = us[:, 1]
    theta_wrap = (theta + np.pi) % (2 * np.pi) - np.pi

    color_t = ts / T_end

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Theta vs. Theta dot',
                        'Theta dot vs. time'),
        vertical_spacing=0.18
    )

    # ── Panel 1: phase portrait ────────────────────────────────────────────────
    fig.add_trace(go.Scatter(x=theta_wrap, y=dtheta,
                             mode='markers',
                             marker=dict(size=2, color=color_t,
                                         colorscale='Plasma',
                                         colorbar=dict(title='t/T', len=0.45,
                                                        y=0.78, x=1.02)),
                             name='phase'), row=1, col=1)

    # ── Panel 2: dtheta(t) ───────────────────────────────────────────────────
    fig.add_trace(go.Scatter(x=ts, y=dtheta, mode='lines',
                             line=dict(color='blue', width=1.2),
                             name='Theta dot(t)'), row=2, col=1)

    fig.update_xaxes(title_text='Theta(rad)', row=1, col=1)
    fig.update_xaxes(title_text='Time(s)', row=2, col=1)
    fig.update_yaxes(title_text='Theta dot(rad/s)', row=1, col=1)
    fig.update_yaxes(title_text='Theta dot(rad/s)', row=2, col=1)

    title = (f'A = {A:.2f} m,  Omega/omega0 = {w_Omega.value:.2f},  '
             f'Theta(initial angle, rad) = {w_theta0.value:.2f},  method = {w_method.value}')
    fig.update_layout(height=620, showlegend=False,
                      title=dict(text=title, font=dict(size=13)),
                      margin=dict(t=80, b=40, l=60, r=80))

    with out:
        out.clear_output(wait=True)
        fig.show()

btn = widgets.Button(description='Run', button_style='primary',
                     layout=widgets.Layout(width='120px', height='36px'))
btn.on_click(run)

left  = widgets.VBox([w_A, w_Omega, w_theta0])
right = widgets.VBox([w_dth0, w_T, w_method])
controls = widgets.VBox([widgets.HBox([left, right]), btn])

display(controls, out)
run()


Output()

In [ ]:
w_A_anim = widgets.Dropdown(options=[(f'{x:.2f}', x) for x in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.75, 1.0, 1.5]],
                             value=0.1, description='A (m)', style=style, layout=widgets.Layout(width='180px'))
out_anim = widgets.Output()

def run_anim(_=None):
    A     = w_A_anim.value if w_A_anim.value is not None else 0.1
    Omega = 2.0 * omega0
    T_end = 15.0
    h     = 0.002

    ts, us = runge_kutta_method(f_pendulum, 0, T_end, [0.2, 0.0], h, A, Omega)
    theta  = us[:, 0]
    stride = max(1, len(ts) // 600)
    ts_d, theta_d = ts[::stride], theta[::stride]

    pivot_y = A * np.cos(Omega * ts_d)      # pivot moves vertically
    bob_x   = l * np.sin(theta_d)
    bob_y   = pivot_y - l * np.cos(theta_d)

    pad     = l * 1.4 + A
    frames = []
    for k in range(len(ts_d)):
        frames.append(go.Frame(
            data=[
                go.Scatter(x=[0, bob_x[k]], y=[pivot_y[k], bob_y[k]],
                           mode='lines', line=dict(color='#31688e', width=3)),
                go.Scatter(x=[0], y=[pivot_y[k]], mode='markers',
                           marker=dict(size=10, color='#21918c')),
                go.Scatter(x=[bob_x[k]], y=[bob_y[k]], mode='markers',
                           marker=dict(size=18, color='#fde725')),
                go.Scatter(x=bob_x[:k+1], y=bob_y[:k+1],
                           mode='lines',
                           line=dict(color='rgba(53,183,121,0.4)', width=1.5))
            ],
            name=str(k),
            layout=go.Layout(title_text=f't = {ts_d[k]:.2f} s  |  θ = {np.degrees(theta_d[k]):.1f}°')
        ))

    fig = go.Figure(
        data=frames[0].data,
        layout=go.Layout(
            title=f'A={A:.2f} m',
            xaxis=dict(range=[-pad, pad], zeroline=True, title='x (m)',
                       scaleanchor='y', scaleratio=1),
            yaxis=dict(range=[-pad, pad], zeroline=True, title='y (m)'),
            height=520, width=540,
            showlegend=False,
            updatemenus=[dict(
                type='buttons', showactive=False,
                buttons=[
                    dict(label='Play',
                         method='animate',
                         args=[None, dict(frame=dict(duration=30, redraw=True),
                                          fromcurrent=True)]),
                    dict(label='Pause',
                         method='animate',
                         args=[[None], dict(frame=dict(duration=0, redraw=False),
                                             mode='immediate')])
                ]
            )],
            sliders=[dict(
                steps=[dict(args=[[f.name],
                                   dict(mode='immediate',
                                        frame=dict(duration=0, redraw=True))],
                            method='animate', label='')
                       for f in frames],
                transition=dict(duration=0),
                x=0.05, y=0, len=0.9,
                currentvalue=dict(visible=False)
            )]
        ),
        frames=frames
    )

    with out_anim:
        out_anim.clear_output(wait=True)
        fig.show()

btn_anim = widgets.Button(description='Animate',
                           layout=widgets.Layout(width='130px', height='36px'))
btn_anim.on_click(run_anim)

display(widgets.HBox([
    widgets.VBox([w_A_anim, btn_anim],
                 layout=widgets.Layout(width='200px', margin='0 20px 0 0')),
    out_anim
], layout=widgets.Layout(align_items='flex-start', width='100%')))

In [ ]:
# 3d spherical pendulum (theta, phi) with vertically driven pivot
def f_spherical_pendulum(t, u, A, Omega):
    # u = [theta, phi, theta_dot, phi_dot]
    theta, phi, th_dot, ph_dot = u[0], u[1], u[2], u[3]
    z_ddot = -A * Omega**2 * np.cos(Omega * t)
    sin_th, cos_th = np.sin(theta), np.cos(theta)
    th_ddot = sin_th * cos_th * ph_dot**2 - (g + z_ddot) / l * sin_th
    # avoid div by zero at theta=0
    ph_ddot = -2 * th_dot * ph_dot * cos_th / (sin_th + 1e-12) if abs(sin_th) > 1e-8 else 0
    return np.array([th_dot, ph_dot, th_ddot, ph_ddot])

w_A_3d  = widgets.FloatSlider(value=0.1, min=0.0, max=1.5, step=0.05,
                               description='A (m)', style=style, layout=widgets.Layout(width='380px'))
w_Or_3d = widgets.FloatSlider(value=2.0, min=0.5, max=4.0, step=0.05,
                               description='Ω/ω₀', style=style, layout=widgets.Layout(width='380px'))
w_T_3d  = widgets.FloatSlider(value=15.0, min=3.0, max=40.0, step=1.0,
                               description='T (s)', style=style, layout=widgets.Layout(width='380px'))
w_phi0  = widgets.FloatSlider(value=0.3, min=0.0, max=np.pi, step=0.05,
                               description='φ₀ (rad)', style=style, layout=widgets.Layout(width='380px'))
out_3d  = widgets.Output()

def run_anim_3d(_=None):
    A     = w_A_3d.value
    Omega = w_Or_3d.value * omega0
    T_end = w_T_3d.value
    phi0  = w_phi0.value
    h     = 0.008

    # u0 = [theta0, phi0, theta_dot0, phi_dot0]; phi0 != 0 gives 3d motion
    u0 = [0.2, phi0, 0.0, 0.0]
    ts, us = runge_kutta_method(f_spherical_pendulum, 0, T_end, u0, h, A, Omega)
    theta_d, phi_d = us[:, 0], us[:, 1]

    stride = max(1, len(ts) // 400)
    ts_d = ts[::stride]
    theta_d = us[::stride, 0]
    phi_d = us[::stride, 1]

    pivot_z = A * np.cos(Omega * ts_d)
    bob_x = l * np.sin(theta_d) * np.cos(phi_d)
    bob_y = l * np.sin(theta_d) * np.sin(phi_d)
    bob_z = pivot_z - l * np.cos(theta_d)

    pad = l * 1.4 + A

    frames = []
    for k in range(len(ts_d)):
        rod_trace = go.Scatter3d(
            x=[0, bob_x[k]], y=[0, bob_y[k]], z=[pivot_z[k], bob_z[k]],
            mode='lines', line=dict(color='#555', width=6)
        )
        pivot_trace = go.Scatter3d(
            x=[0], y=[0], z=[pivot_z[k]],
            mode='markers', marker=dict(size=8, color='steelblue', symbol='diamond')
        )
        bob_trace = go.Scatter3d(
            x=[bob_x[k]], y=[bob_y[k]], z=[bob_z[k]],
            mode='markers', marker=dict(size=14, color='tomato')
        )
        trail_trace = go.Scatter3d(
            x=bob_x[:k+1], y=bob_y[:k+1], z=bob_z[:k+1],
            mode='lines', line=dict(color='rgba(220,100,80,0.4)', width=2)
        )
        frames.append(go.Frame(
            data=[rod_trace, pivot_trace, bob_trace, trail_trace],
            name=str(k),
            layout=go.Layout(title_text=f't = {ts_d[k]:.2f} s  |  θ = {np.degrees(theta_d[k]):.1f}°  φ = {np.degrees(phi_d[k]):.1f}°')
        ))

    fig = go.Figure(
        data=frames[0].data,
        layout=go.Layout(
            title=f'3D pendulum  A={A:.2f} m, Ω/ω₀={w_Or_3d.value:.2f}',
            scene=dict(
                xaxis=dict(range=[-pad, pad], title='x'),
                yaxis=dict(range=[-pad, pad], title='y'),
                zaxis=dict(range=[-pad, pad], title='z'),
                aspectmode='data'
            ),
            height=560, width=560,
            showlegend=False,
            updatemenus=[dict(
                type='buttons', showactive=False,
                buttons=[
                    dict(label='Play', method='animate',
                         args=[None, dict(frame=dict(duration=30, redraw=True), fromcurrent=True)]),
                    dict(label='Pause', method='animate',
                         args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')])
                ]
            )],
            sliders=[dict(
                steps=[dict(args=[[f.name], dict(mode='immediate', frame=dict(duration=0, redraw=True))],
                            method='animate', label='') for f in frames],
                transition=dict(duration=0), x=0.05, y=0, len=0.9,
                currentvalue=dict(visible=False)
            )]
        ),
        frames=frames
    )

    with out_3d:
        out_3d.clear_output(wait=True)
        fig.show()

btn_3d = widgets.Button(description='Animate 3D',
                        layout=widgets.Layout(width='130px', height='36px'))
btn_3d.on_click(run_anim_3d)

display(widgets.VBox([
    widgets.HBox([w_A_3d, w_Or_3d, w_T_3d, w_phi0]),
    btn_3d
]), out_3d)

Output()

## Simulating strings with wave equation

The wave equation is given to us by:
$$\frac{\partial^2u}{\partial t^2} = v^2 \frac{\partial^2 u}{\partial x^2} + \gamma \frac{\partial}{\partial t} \frac{\partial^2 u}{\partial t^2}$$

We can assume there are fixed boundary conditions of the string being fixed on both ends (e.g., a violin string), which means that the string is set at $u(0,t) = u(1,t) = 0$

We can say that the "pluck" is going to happen at position x:

$$u(x,0) = \begin{cases} h\,x/x_p & x \le x_p \\ h\,(1-x)/(1-x_p) & x > x_p \end{cases}, \qquad \dot{u}(x,0) = 0.$$

Therefore, from that point x, we would propogate the pluck along the string. 

We can rewrite the wave equation as a system of equations

$$\frac{\partial^2 u}{\partial t^2}\bigg|^n_j \approx \frac{u^{n+1}_j - 2u^n_j + u^{n-1}_j}{\Delta t^2}, \qquad \frac{\partial^2 u}{\partial x^2}\bigg|^n_j \approx \frac{u^n_{j+1} - 2u^n_j + u^n_{j-1}}{\Delta x^2} \equiv \frac{(\delta^2_x u^n)_j}{\Delta x^2}.$$

For the mixed derivative $\partial_t\partial^2_x u$, apply a central difference in time to $\partial^2_x u$:

$$\frac{\partial}{\partial t}\frac{\partial^2 u}{\partial x^2}\bigg|^n_j \approx \frac{(\delta^2_x u^{n+1})_j - (\delta^2_x u^{n-1})_j}{2\,\Delta t\,\Delta x^2}.$$


Let $\mathbf{u}^n = (u^n_1, \ldots, u^n_N)^\top$ be the vector of interior values and $L$ the $N\times N$ tridiagonal second-difference matrix:

$$L = \begin{pmatrix} -2 & 1 & & \\ 1 & -2 & 1 & \\ & \ddots & \ddots & \ddots \\ & & 1 & -2 \end{pmatrix}.$$

$${(I - sL)\,\mathbf{u}^{n+1} = 2\mathbf{u}^n - \mathbf{u}^{n-1} + r^2 L\mathbf{u}^n + sL\mathbf{u}^{n-1}.} \tag{3}$$

The starting step should then be:
$$\mathbf{u}^1 = (I - sL)^{-1}\!\left[\mathbf{u}^0 + \tfrac{r^2}{2}L\mathbf{u}^0\right].$$

I wanted to model this visually to see what would happen

What happens if we add material properties to the string?

What happens if we have multiple "plucks" happening within short sequence of eachother?

In [ ]:
def make_L(N):
    """Tridiagonal second-difference matrix (N interior points, Dirichlet BCs)."""
    L = np.zeros((N, N))
    np.fill_diagonal(L, -2)
    np.fill_diagonal(L[1:], 1)
    np.fill_diagonal(L[:, 1:], 1)
    return L

def pluck(x, x_peak, height):
    return np.where(x <= x_peak,
                    height * x / x_peak,
                    height * (1 - x) / (1 - x_peak))

def solve_string(v=1.0, gamma=0.0, Nx=120, T=6.0, CFL=0.9,
                 x_peak=0.3, height=1.0, n_frames=400):
    """
    Solve  u_tt = v² u_xx + γ u_xxt  on [0,1] with Dirichlet BCs.
    Returns:
        ts     : (n_frames,)     saved times
        us     : (n_frames, Nx)  saved displacement fields
        x      : (Nx,)           interior grid points
        energy : (n_frames,)     total mechanical energy
    """
    dx  = 1.0 / (Nx + 1)
    dt  = CFL * dx / v
    Nt  = int(T / dt)
    r   = v * dt / dx          
    s   = gamma * dt / (2 * dx**2)  # damping parameter

    x   = np.linspace(dx, 1 - dx, Nx)
    L   = make_L(Nx)
    A   = np.eye(Nx) - s * L   # LHS system matrix (tridiagonal)
    u0      = pluck(x, x_peak, height)
    u_prev  = u0.copy()
    u_curr  = np.linalg.solve(A, u0 + 0.5 * r**2 * (L @ u0))
    def energy(u_m, u_p):  # kinetic + potential
        dudt  = (u_p - u_m) / (2 * dt)
        dudx  = np.gradient(np.r_[0, u_m, 0], dx)
        return 0.5 * dx * (np.sum(dudt**2) + v**2 * np.sum(dudx**2))

    # ── Time-march ────────────────────────────────────────────────────────
    stride  = max(1, Nt // n_frames)
    ts_out, us_out, en_out = [0.0], [u0.copy()], [energy(u0, u_curr)]

    for n in range(1, Nt):
        if np.any(np.abs(u_curr) > 1e10) or np.any(~np.isfinite(u_curr)):
            break
        rhs    = (2 * u_curr - u_prev
                  + r**2 * (L @ u_curr)
                  + s   * (L @ u_prev))
        u_next = np.linalg.solve(A, rhs)
        u_prev, u_curr = u_curr, u_next

        if n % stride == 0:
            ts_out.append(n * dt)
            us_out.append(u_curr.copy())
            en_out.append(energy(u_prev, u_curr))

    return (np.array(ts_out), np.array(us_out),
            x, np.array(en_out))

In [19]:
fig = make_subplots(1, 2, subplot_titles=(
    'Stable: CFL = 0.9  (r < 1)',
    'Unstable: CFL = 1.01  (r > 1)'))

for col, cfl in enumerate([0.9, 1.01], start=1):
    try:
        ts, us, x, _ = solve_string(CFL=cfl, T=3.0, n_frames=200)
        # plot 5 snapshots
        idxs = np.linspace(0, len(ts)-1, 5, dtype=int)
        for i, idx in enumerate(idxs):
            u = us[idx]
            if np.max(np.abs(u)) > 1e6:
                u = np.full_like(u, np.nan)
            fig.add_trace(go.Scatter(
                x=x, y=u, mode='lines',
                line=dict(width=1.5),
                name=f't={ts[idx]:.2f}',
                showlegend=(col==1)
            ), row=1, col=col)
    except Exception:
        pass

fig.add_annotation(text='Blows up — solution not shown',
                   xref='x2', yref='y2', x=0.5, y=0,
                   showarrow=False, font=dict(color='red', size=13))
fig.update_yaxes(range=[-1.5, 1.5])
fig.update_xaxes(title_text='x')
fig.update_yaxes(title_text='u(x,t)', col=1)
fig.update_layout(height=380, title='CFL Stability Comparison', margin=dict(t=70))
fig.show()

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:38: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:38: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:38: RuntimeWarning:

invalid value encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:52: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:52: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:52: RuntimeWarning:

invalid value encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:53: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T

In [20]:
T_sim = 10.0
ts0, us0, x, en0 = solve_string(gamma=0.0,   T=T_sim, n_frames=500)
ts1, us1, _, en1 = solve_string(gamma=0.005, T=T_sim, n_frames=500)
ts2, us2, _, en2 = solve_string(gamma=0.02,  T=T_sim, n_frames=500)

fig = make_subplots(1, 2,
    subplot_titles=('Energy vs time', 'Snapshot at t = 5 s'),
    horizontal_spacing=0.12)

for ts, en, label, col in zip(
        [ts0, ts1, ts2], [en0, en1, en2],
        ['γ=0', 'γ=0.005', 'γ=0.02'],
        ['royalblue', 'tomato', 'seagreen']):
    fig.add_trace(go.Scatter(x=ts, y=en/en[0], mode='lines',
                             name=label,
                             line=dict(color=col, width=2)), row=1, col=1)

idx5 = np.argmin(np.abs(ts0 - 5.0))
for us, label, col in zip(
        [us0, us1, us2],
        ['γ=0', 'γ=0.005', 'γ=0.02'],
        ['royalblue', 'tomato', 'seagreen']):
    fig.add_trace(go.Scatter(x=x, y=us[idx5], mode='lines',
                             name=label, showlegend=False,
                             line=dict(color=col, width=2)), row=1, col=2)

fig.update_xaxes(title_text='t (s)', row=1, col=1)
fig.update_xaxes(title_text='x', row=1, col=2)
fig.update_yaxes(title_text='E(t) / E(0)', row=1, col=1)
fig.update_yaxes(title_text='u(x, 5)', row=1, col=2)
fig.update_layout(height=400, title='Effect of Kelvin–Voigt Damping', margin=dict(t=60))
fig.show()

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:38: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:38: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:38: RuntimeWarning:

invalid value encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:52: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:52: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:52: RuntimeWarning:

invalid value encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:53: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T

In [21]:
def harmonic_amplitudes(ts, us, N_modes=8):
    """Project u(t) onto sin(nπx) modes; return amplitude vs time for each mode."""
    Nx = us.shape[1]
    x  = np.linspace(1/(Nx+1), Nx/(Nx+1), Nx)
    amps = np.zeros((len(ts), N_modes))
    us_safe = np.nan_to_num(us, nan=0.0, posinf=0.0, neginf=0.0)
    for n in range(1, N_modes+1):
        mode = np.sin(n * np.pi * x)
        proj = us_safe @ mode / (Nx / 2)  # inner product, normalised
        amps[:, n-1] = np.abs(proj)
    return amps

N_modes = 6
amps0 = harmonic_amplitudes(ts0, us0, N_modes)
amps2 = harmonic_amplitudes(ts2, us2, N_modes)

colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b']

fig = make_subplots(1, 2,
    subplot_titles=('Undamped  (γ=0)', 'Damped  (γ=0.02)'),
    shared_yaxes=True)

for n in range(N_modes):
    kw = dict(mode='lines', name=f'n={n+1}',
              line=dict(color=colors[n], width=1.8))
    fig.add_trace(go.Scatter(x=ts0, y=amps0[:, n], **kw),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=ts2, y=amps2[:, n],
                             showlegend=False,
                             line=dict(color=colors[n], width=1.8),
                             mode='lines'),
                  row=1, col=2)

fig.update_xaxes(title_text='t (s)')
fig.update_yaxes(title_text='Modal amplitude', col=1)
fig.update_layout(height=420, title='Harmonic Amplitudes vs Time',
                  margin=dict(t=60))
fig.show()

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/2521398291.py:8: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/2521398291.py:8: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/2521398291.py:8: RuntimeWarning:

invalid value encountered in matmul



In [ ]:
style  = {'description_width': '160px'}
layout = widgets.Layout(width='460px')

w_v      = widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1,
                                description='v  (wave speed)',     style=style, layout=layout)
w_gamma  = widgets.FloatSlider(value=0.0, min=0.0, max=0.05, step=0.001,
                                description='γ  (damping)',         style=style, layout=layout,
                                readout_format='.3f')
w_xpeak  = widgets.FloatSlider(value=0.3, min=0.05, max=0.95, step=0.05,
                                description='x_peak (pluck pos.)',  style=style, layout=layout)
w_CFL    = widgets.FloatSlider(value=0.9, min=0.5,  max=0.99, step=0.01,
                                description='CFL  (step ratio)',    style=style, layout=layout)
w_T      = widgets.FloatSlider(value=6.0, min=2.0,  max=20.0, step=1.0,
                                description='T  (end time, s)',     style=style, layout=layout)
out      = widgets.Output()

def run(_=None):
    v     = w_v.value
    gamma = w_gamma.value
    xp    = w_xpeak.value
    cfl   = w_CFL.value
    T_end = w_T.value

    if cfl >= 1.0:
        with out:
            out.clear_output(wait=True)
            print('⚠  CFL > 1: scheme is unstable — reduce step ratio.')
        return

    ts, us, x, en = solve_string(
        v=v, gamma=gamma, CFL=cfl, T=T_end,
        x_peak=xp, n_frames=350)

    # Spectral amplitudes
    N_modes = 6
    amps    = harmonic_amplitudes(ts, us, N_modes)

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('u(x, t)  — 8 snapshots',
                        'Phase-space  (u, u_t)  at x = x_peak',
                        'Energy  E(t) / E(0)',
                        'Harmonic amplitudes'),
        vertical_spacing=0.16, horizontal_spacing=0.10
    )

    # ── Panel 1: snapshots ────────────────────────────────────────────────
    snap_idx = np.linspace(0, len(ts)-1, 8, dtype=int)
    for k, idx in enumerate(snap_idx):
        alpha = 0.4 + 0.6 * k / 7
        fig.add_trace(go.Scatter(
            x=x, y=us[idx], mode='lines',
            line=dict(color=f'rgba(31,119,180,{alpha:.2f})', width=1.5),
            name=f't={ts[idx]:.1f}'), row=1, col=1)
    # Mark pluck position
    fig.add_vline(x=xp, line=dict(color='red', dash='dot', width=1),
                  row=1, col=1)

    # ── Panel 2: phase-space at x_peak ────────────────────────────────────
    j_peak = np.argmin(np.abs(x - xp))
    u_phase  = us[:, j_peak]
    dt_local = np.mean(np.diff(ts))
    dudt     = np.gradient(u_phase, dt_local)
    fig.add_trace(go.Scatter(
        x=u_phase, y=dudt, mode='lines',
        line=dict(color='tomato', width=1.2),
        showlegend=False), row=1, col=2)

    # ── Panel 3: energy ───────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=ts, y=en / en[0], mode='lines',
        line=dict(color='seagreen', width=2),
        showlegend=False), row=2, col=1)

    # ── Panel 4: harmonic amplitudes ──────────────────────────────────────
    for n in range(N_modes):
        fig.add_trace(go.Scatter(
            x=ts, y=amps[:, n], mode='lines',
            name=f'n={n+1}',
            line=dict(color=colors[n], width=1.5)), row=2, col=2)

    fig.update_xaxes(title_text='x',    row=1, col=1)
    fig.update_xaxes(title_text='u',    row=1, col=2)
    fig.update_xaxes(title_text='t (s)',row=2, col=1)
    fig.update_xaxes(title_text='t (s)',row=2, col=2)
    fig.update_yaxes(title_text='u',    row=1, col=1)
    fig.update_yaxes(title_text='u_t',  row=1, col=2)
    fig.update_yaxes(title_text='E/E₀', row=2, col=1)
    fig.update_yaxes(title_text='|aₙ|', row=2, col=2)

    fig.update_layout(
        height=660, showlegend=True,
        title=dict(text=f'v={v:.1f}, γ={gamma:.3f}, x_peak={xp:.2f}, CFL={cfl:.2f}',
                   font=dict(size=13)),
        legend=dict(x=1.02, y=0.25),
        margin=dict(t=80, b=40, l=60, r=120)
    )

    with out:
        out.clear_output(wait=True)
        fig.show()

btn = widgets.Button(description='Run', button_style='primary',
                     layout=widgets.Layout(width='110px', height='36px'))
btn.on_click(run)

left  = widgets.VBox([w_v, w_gamma, w_xpeak])
right = widgets.VBox([w_CFL, w_T])
display(widgets.VBox([widgets.HBox([left, right]), btn]), out)
run()

Output()

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:38: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:38: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:38: RuntimeWarning:

invalid value encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:52: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:52: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:52: RuntimeWarning:

invalid value encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:53: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T

In [23]:
w_A_g  = widgets.FloatSlider(value=0.005, min=0.0, max=0.05, step=0.005,
                              description='γ', readout_format='.3f',
                              style=style, layout=widgets.Layout(width='380px'))
w_A_xp = widgets.FloatSlider(value=0.3, min=0.05, max=0.95, step=0.05,
                              description='x_peak',
                              style=style, layout=widgets.Layout(width='380px'))
out_anim = widgets.Output()

def run_anim(_=None):
    ts, us, x, _ = solve_string(
        gamma=w_A_g.value, T=5.0, CFL=0.9,
        x_peak=w_A_xp.value, n_frames=250)

    x_full = np.r_[0, x, 1]

    frames = []
    for k in range(len(ts)):
        u_full = np.r_[0, us[k], 0]
        frames.append(go.Frame(
            data=[go.Scatter(x=x_full, y=u_full, mode='lines',
                             line=dict(color='royalblue', width=2.5))],
            name=str(k),
            layout=go.Layout(title_text=f't = {ts[k]:.3f} s')
        ))

    u0_full = np.r_[0, us[0], 1*0]
    fig = go.Figure(
        data=[go.Scatter(x=x_full, y=u0_full, mode='lines',
                         line=dict(color='royalblue', width=2.5))],
        layout=go.Layout(
            title=f'γ={w_A_g.value:.3f}, x_peak={w_A_xp.value:.2f}',
            xaxis=dict(range=[0, 1], title='x'),
            yaxis=dict(range=[-1.3, 1.3], title='u(x,t)'),
            height=400, width=700,
            showlegend=False,
            updatemenus=[dict(
                type='buttons', showactive=False,
                buttons=[
                    dict(label='▶ Play', method='animate',
                         args=[None, dict(frame=dict(duration=25, redraw=True),
                                          fromcurrent=True)]),
                    dict(label='⏸ Pause', method='animate',
                         args=[[None], dict(frame=dict(duration=0, redraw=False),
                                             mode='immediate')])
                ]
            )],
            sliders=[dict(
                steps=[dict(args=[[f.name],
                                   dict(mode='immediate',
                                        frame=dict(duration=0, redraw=True))],
                            method='animate', label='') for f in frames],
                x=0.05, y=0, len=0.9,
                currentvalue=dict(visible=False),
                transition=dict(duration=0)
            )]
        ),
        frames=frames
    )

    with out_anim:
        out_anim.clear_output(wait=True)
        fig.show()

btn_anim = widgets.Button(description='Animate', button_style='success',
                           layout=widgets.Layout(width='120px', height='36px'))
btn_anim.on_click(run_anim)
display(widgets.VBox([widgets.HBox([w_A_g, w_A_xp]), btn_anim]), out_anim)
run_anim()

Output()

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:38: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:38: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:38: RuntimeWarning:

invalid value encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:52: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:52: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:52: RuntimeWarning:

invalid value encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/4267162330.py:53: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T